# Loan Approval Prediction

**Client scenario:** A bank wants to automate its loan approval process by 
predicting whether an applicant's loan should be approved or rejected, based 
on their financial profile (income, credit history, loan amount) and personal 
details (marital status, education, property area).

In [123]:
import pandas as pd
df=pd.read_csv('train.csv')
print("lenght=",len(df))
print(df.head(614))


lenght= 614
      Loan_ID  Gender Married Dependents     Education Self_Employed  \
0    LP001002    Male      No          0      Graduate            No   
1    LP001003    Male     Yes          1      Graduate            No   
2    LP001005    Male     Yes          0      Graduate           Yes   
3    LP001006    Male     Yes          0  Not Graduate            No   
4    LP001008    Male      No          0      Graduate            No   
..        ...     ...     ...        ...           ...           ...   
609  LP002978  Female      No          0      Graduate            No   
610  LP002979    Male     Yes         3+      Graduate            No   
611  LP002983    Male     Yes          1      Graduate            No   
612  LP002984    Male     Yes          2      Graduate            No   
613  LP002990  Female      No          0      Graduate           Yes   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0               5849                0.0         Na

In [124]:
baseline_accuracy=(df['Loan_Status']=='Y').mean()
print('B-A',baseline_accuracy)
df['LoanAmount']=df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term']=df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())
df['Credit_History']=df['Credit_History'].fillna(df['Credit_History'].median())
df['Gender']=df['Gender'].fillna(df['Gender'].mode()[0])
df['Married']=df['Married'].fillna(df['Married'].mode()[0])
df['Dependents']=df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed']=df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
print(df.isnull().sum())

B-A 0.6872964169381107
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


In [125]:
df['Gender']=df['Gender'].map({'Male':1 ,'Female':0})
df['Married']=df['Married'].map({'Yes':1 ,'No':0})
df['Education']=df['Education'].map({'Graduate':1 ,'Not Graduate':0})
df['Self_Employed']=df['Self_Employed'].map({'Yes':1 ,'No':0})
df['Loan_Status']=df['Loan_Status'].map({'Y':1 ,'N':0})
df['Dependents']=df['Dependents'].replace('3+','3')
df['Dependents']=pd.to_numeric(df['Dependents'])


In [126]:
df['Property_Area'] = df['Property_Area'].map({'Urban': 0, 'Semiurban': 1, 'Rural': 2})

In [127]:
print(df.columns.tolist())

['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


In [128]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
x= df[['Credit_History', 'ApplicantIncome', 'LoanAmount', 'Gender', 'Married', 'Education', 'Property_Area']]
y = df['Loan_Status']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
model=DecisionTreeClassifier(random_state=42,max_depth=8)
model.fit(x_train,y_train)
prediction=model.predict(x_test)
model_accuracy=accuracy_score(y_test,prediction)
print('model_accuracy:',model_accuracy)
print('baseline_accuracy:',baseline_accuracy)
print('improvemnt over baqseline:',model_accuracy-baseline_accuracy)

model_accuracy: 0.7398373983739838
baseline_accuracy: 0.6872964169381107
improvemnt over baqseline: 0.052540981435873046


In [129]:
results=x_test.copy()
results['actual']=y_test
results['predicted']=prediction

wrong_prediction=results[results['actual'] != results['predicted']]
print('wrong_prediction length',len(wrong_prediction))
print(wrong_prediction.head())

wrong_prediction length 32
     Credit_History  ApplicantIncome  LoanAmount  Gender  Married  Education  \
340             1.0             2647       173.0       1        1          0   
77              1.0             1000       110.0       1        1          1   
209             1.0             3418       127.0       1        0          1   
610             1.0             4106        40.0       1        1          1   
148             1.0            10000       225.0       0        0          1   

     Property_Area  actual  predicted  
340              2       0          1  
77               0       0          1  
209              1       0          1  
610              2       1          0  
148              2       0          1  


In [130]:
importances = pd.Series(model.feature_importances_, index=x.columns)
print(importances.sort_values(ascending=False))

Credit_History     0.453013
LoanAmount         0.270602
ApplicantIncome    0.184136
Gender             0.036437
Property_Area      0.032743
Married            0.019592
Education          0.003477
dtype: float64


In [131]:
print("Model accuracy:", model_accuracy)


Model accuracy: 0.7398373983739838


In [132]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler


In [133]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
baseline=y_test.value_counts(normalize=True).max()
tree=DecisionTreeClassifier(max_depth=4,random_state=42).fit(x_train,y_train)
tree_accuracy=accuracy_score(y_test,tree.predict(x_test))
print('baseline',baseline)                             
print('tree accuracy:',tree_accuracy)                             

baseline 0.6504065040650406
tree accuracy: 0.7560975609756098


In [134]:
knn=KNeighborsClassifier(n_neighbors=3).fit(x_train,y_train)
unscaled_knn=accuracy_score(y_test,knn.predict(x_test))
print('unscaled_knn',unscaled_knn)



unscaled_knn 0.5853658536585366


In [135]:
scaler=StandardScaler()
x_train_s=scaler.fit_transform(x_train)
x_test_s=scaler.transform(x_test)
knn_s=KNeighborsClassifier(n_neighbors=5).fit(x_train_s,y_train)
scaled_knn=accuracy_score(y_test,knn_s.predict(x_test_s))
print('scaled knn',scaled_knn)


scaled knn 0.7886178861788617


In [136]:
for k in [1,3,5,7,9,14]:
    m=KNeighborsClassifier(n_neighbors=k).fit(x_train_s,y_train)
    print(k,round(m.score(x_train_s,y_train),3),round(m.score(x_test_s,y_test)))

1 1.0 1
3 0.859 1
5 0.837 1
7 0.817 1
9 0.809 1
14 0.811 1


In [137]:
import pandas as pd
data={
    'model':['baseline','DecisionTree','scaled_knn','unscaled_knn'],
    'Accuracy':[baseline,tree_accuracy,unscaled_knn,scaled_knn]
}
accuracy_scores=pd.DataFrame(data)
print(accuracy_scores)

          model  Accuracy
0      baseline  0.650407
1  DecisionTree  0.756098
2    scaled_knn  0.585366
3  unscaled_knn  0.788618


In [138]:
from sklearn.metrics import confusion_matrix
print('connfusion-matrix of tree',confusion_matrix(y_test,tree.predict(x_test)))
print('connfusion-matrix of knn_s',confusion_matrix(y_test,tree.predict(x_test_s)))

connfusion-matrix of tree [[16 27]
 [ 3 77]]
connfusion-matrix of knn_s [[43  0]
 [80  0]]


c:\Users\Qadri Laptop\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


In [139]:
from sklearn.svm import SVC
svm_linear=SVC(kernel='linear',random_state=42)
svm_linear.fit(x_train_s,y_train)
svm_linear_acc=accuracy_score(y_test,svm_linear.predict(x_test_s))
print(svm_linear_acc)



0.7886178861788617


In [140]:
svm_rbf=SVC(kernel='rbf',random_state=42)
svm_rbf.fit(x_train_s,y_train)
svm_rbf_acc=accuracy_score(y_test,svm_rbf.predict(x_test_s))
print(svm_rbf_acc)




0.7967479674796748


In [141]:
table=pd.DataFrame(
    {
        'model':['baseline','DecisionTree','scaled_knn','unscaled_knn','svm_linear','svm_rbf'],
        'accuracy_score':[baseline,tree_accuracy,scaled_knn,unscaled_knn,svm_linear_acc,svm_rbf_acc]

    }
)
print(table)

          model  accuracy_score
0      baseline        0.650407
1  DecisionTree        0.756098
2    scaled_knn        0.788618
3  unscaled_knn        0.585366
4    svm_linear        0.788618
5       svm_rbf        0.796748


In [142]:
from sklearn.model_selection import cross_val_score
tree=DecisionTreeClassifier(max_depth=4,random_state=42)
scores=cross_val_score(tree,x,y,cv=5)
print(scores)

[0.7804878  0.7398374  0.75609756 0.82926829 0.81967213]


In [143]:
print('mean:',scores.mean())
print('std:',scores.std())

mean: 0.7850726376116219
std: 0.03480569926396492


In [144]:
models={
    'DecisionTree':DecisionTreeClassifier(max_depth=4,random_state=42),
    'knn':KNeighborsClassifier(n_neighbors=5),
    'svm_linear':SVC(kernel='linear',random_state=42),
    'svm_rbf':SVC(kernel='rbf'),
}
for name,model in models.items():
    scores=cross_val_score(model,x_train_s,y_train,cv=5)
    print(name,'mean:',round(scores.mean(),3),'std:',round(scores.std(),3))

DecisionTree mean: 0.792 std: 0.011
knn mean: 0.79 std: 0.023
svm_linear mean: 0.815 std: 0.021
svm_rbf mean: 0.809 std: 0.014


In [145]:
print(df.columns)

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='str')


In [146]:
print(type(df))
print(df.columns.tolist())

<class 'pandas.DataFrame'>
['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']


['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']

In [ ]:
#Day 25 : Feature Engineering
df['loan_per_income']=df['LoanAmount'] / df['ApplicantIncome']
print(df['loan_per_income'])
df['totalincome']=df['ApplicantIncome']+df['CoapplicantIncome']
print(df['totalincome'])
df['income_per_dependent']=df['totalincome']/(df['Dependents']+1)
print(df['income_per_dependent'])


0      0.021884
1      0.027929
2      0.022000
3      0.046458
4      0.023500
         ...   
609    0.024483
610    0.009742
611    0.031343
612    0.024660
613    0.029020
Name: loan_per_income, Length: 614, dtype: float64
0      5849.0
1      6091.0
2      3000.0
3      4941.0
4      6000.0
        ...  
609    2900.0
610    4106.0
611    8312.0
612    7583.0
613    4583.0
Name: totalincome, Length: 614, dtype: float64
0      5849.000000
1      3045.500000
2      3000.000000
3      4941.000000
4      6000.000000
          ...     
609    2900.000000
610    1026.500000
611    4156.000000
612    2527.666667
613    4583.000000
Name: income_per_dependent, Length: 614, dtype: float64


In [148]:
print(df.columns)

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status',
       'loan_per_income', 'totalincome', 'income_per_dependent'],
      dtype='str')


In [154]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
x_old=df[['ApplicantIncome','CoapplicantIncome','LoanAmount','Credit_History']]
x_new=df[['ApplicantIncome','CoapplicantIncome','LoanAmount','Credit_History','loan_per_income','totalincome','income_per_dependent']]
y=df['Loan_Status']
model=DecisionTreeClassifier(max_depth=4,random_state=42)
score_old=cross_val_score(model,x_old,y,cv=5)
score_new=cross_val_score(model,x_new,y,cv=5)
print('old_mean:',round(score_old.mean(),3))
print('new_mean:',round(score_new.mean(),3))


old_mean: 0.801
new_mean: 0.792
